<div dir="rtl">
<h1>نسخهٔ دوم را از قطعه‌های خودش بازسازی کنید</h1>
<p>درس 38 از 76 · Self-Attention و نسخهٔ دو پروژه · <code dir="ltr">32-self</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/32-self.html">📖 بازگشت به همین درس</a></p>
<p>مسیر تک‌سر واقعی v2 را مونتاژ و تفاوت وزن Attention با Logits را مشاهده کنید.</p><p>پیش‌نیاز: Embedding، Q/K/V، امتیاز مقیاس‌شده و ترکیب Value را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>با Vocabulary ۱۲تایی و ورودی چهارموقعیتی، وزن Attention و خروجی مدل چه شکل‌هایی دارند؟ آیا واژهٔ Self یعنی فقط قطر جدول مجاز است؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.stages.v2 import SingleHead
model = SingleHead(12,channels=6).eval()
ids = torch.tensor([[1,2,3,4]])
with torch.no_grad():
    reference = model(ids)
print('logits shape:',reference.shape,'weights shape:',model.last_weights.shape)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع assemble_v2(Model, ids) را با Embedding، Query، Key، Value و Head همین شیء بسازید؛ خود model(ids) را در پاسخ صدا نزنید. زوج (Logits, weights) برگردانید. v2 در این دفتر بدون Mask است.</p>
</div>

In [ ]:
def assemble_v2(model, ids):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = assemble_v2(model,ids)
    if result is None: return False
    for tokens in (ids,torch.tensor([[4,2,1],[2,2,3]])):
        logits,weights = assemble_v2(model,tokens)
        expected = model(tokens)
        torch.testing.assert_close(logits,expected)
        torch.testing.assert_close(weights,model.last_weights)
        assert logits.shape[-1] == 12
        torch.testing.assert_close(weights.sum(-1),torch.ones_like(weights.sum(-1)))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط دو Token آینده را عوض کنید؛ همان مدل با همان وزن‌ها را نگه دارید. اختلاف پیشوند، امکان دریافت اطلاعات آینده را نشان می‌دهد نه کیفیت زبان را.</p>
</div>

In [ ]:
changed = ids.clone()
changed[:,2:] = torch.tensor([9,10])
with torch.no_grad():
    print('prefix change in unmasked v2:',(model(ids)[:,:2]-model(changed)[:,:2]).abs().max().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>پیاده‌سازی خراب، Self را به معنی «فقط خود موقعیت» گرفته است. تابع self_weights(q,k) را اصلاح کنید: منبع Q و K یکی است، اما همهٔ جفت‌ها در این نسخهٔ بدون Mask مجازند.</p>
</div>

In [ ]:
wrong_weights = torch.eye(ids.shape[1])[None]
print('diagonal-only interpretation:',wrong_weights)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def self_weights(q, k):
    # TODO
    return None

In [ ]:
def test_repair():
    q,k = torch.zeros(1,3,2),torch.zeros(1,3,2)
    result = self_weights(q,k)
    if result is None: return False
    torch.testing.assert_close(result,torch.full((1,3,3),1/3))
    q,k = torch.ones(1,2,4),torch.arange(8.).reshape(1,2,4)
    torch.testing.assert_close(self_weights(q,k),(q@k.transpose(-2,-1)/2).softmax(-1))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>تمام Parameterها از SingleHead واقعی گرفته شدند؛ کپی دیگری از مدل تولیدی نساختیم. last_weights ابزار مشاهده است، در حالی که forward این کلاس Logits را برمی‌گرداند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا Self-Attention به‌تنهایی هیچ تضمینی دربارهٔ ندیدن آینده نمی‌دهد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/32-self.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/32-self.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>